# SmileAlign v2, lanjutkan latihan di Kaggle

## Sebelum mulai

1. Kaggle: **Settings > Accelerator: GPU P100 atau T4**, **Internet: On**.
2. Di Drive, dua file ini harus **Anyone with the link**:
   - `smilealign/runs-smilealign-seg-v2/<run>/weights/last.pt`
   - `smilealign/intraoral_yolov8.zip`
3. Ambil ID dari link, bagian antara `/d/` dan `/view`.

## Aturan yang membuat notebook ini jalan

- **Ultralytics dipasang di Sel 1, sebelum checkpoint dibaca.** Membaca `.pt`
  dengan `torch.load` butuh paket ultralytics sudah terpasang, karena isi
  pickle-nya merujuk kelas `ultralytics.nn.tasks.SegmentationModel`. Ini
  kesalahan di versi sebelumnya.
- **Versi tidak dipin.** Kalau versi terpasang berbeda dari pembuat checkpoint,
  Sel 5 menyelaraskan argumennya: kunci asing dibuang, tipe boolean disamakan.
- **Path dataset harus `/content/datasets/mix_v2`** karena path itu tersimpan di
  checkpoint. Sel 7 membangunnya di sana.
- **Path folder hasil harus `/content/drive/MyDrive/smilealign/runs-smilealign-seg-v2`**
  karena itu juga tersimpan di checkpoint. Sel 2 membuat symlink ke folder kerja.
- **Pakai `last.pt`, bukan `best.pt`.** `best.pt` tidak menyimpan state optimizer
  dan nomor epoch.

Jalankan Sel 0 sampai Sel 9 berurutan. Tidak ada sel yang boleh dilewati.

## Sel 0, konfigurasi

In [ ]:
# Tempel URL atau ID Drive, dua-duanya diterima.
# URL : https://drive.google.com/file/d/1AbCdEfGhIjKlMnOpQrStUvWxYz/view?usp=sharing
# ID  : 1AbCdEfGhIjKlMnOpQrStUvWxYz

FILE_LAST_PT = ""
FILE_INTRAORAL_ZIP = ""

import re
from pathlib import Path


def drive_id(value):
    # Terima URL Drive atau ID polos, kembalikan ID polos.
    value = (value or "").strip()
    for pattern in (r"/d/([A-Za-z0-9_-]{10,})", r"[?&]id=([A-Za-z0-9_-]{10,})"):
        m = re.search(pattern, value)
        if m:
            return m.group(1)
    return value


LAST_PT_ID = drive_id(FILE_LAST_PT)
ZIP_ID = drive_id(FILE_INTRAORAL_ZIP)

ROOT = Path("/content/drive/MyDrive/smilealign")
RUNS = ROOT / "runs-smilealign-seg-v2"
STAGING = Path("/kaggle/working/_incoming")
MIX_DIR = Path("/content/datasets/mix_v2")

print("last.pt id :", LAST_PT_ID or "(kosong)")
print("zip id     :", ZIP_ID or "(kosong)")
print("siap       :", bool(LAST_PT_ID and ZIP_ID))

## Sel 1, pasang paket

Ultralytics dipasang di sini, bukan nanti. Tanpa paket ini `torch.load` pada
Sel 4 gagal dengan `ModuleNotFoundError: No module named 'ultralytics'`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

import subprocess
import sys


def pip_install(*specs):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *specs]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print("GAGAL:", " ".join(specs))
        print(r.stdout[-1500:])
        print(r.stderr[-1500:])
    return r.returncode == 0


assert pip_install("-U", "ultralytics"), "ultralytics gagal dipasang"
assert pip_install("gdown", "huggingface_hub"), "paket dasar gagal dipasang"

import torch
import ultralytics

print("torch       :", torch.__version__)
print("cuda        :", torch.cuda.is_available())
print("ultralytics :", ultralytics.__version__)
assert torch.cuda.is_available(), "GPU belum aktif, ubah Settings > Accelerator"

## Sel 2, hidupkan path Drive

Checkpoint menyimpan `project: /content/drive/MyDrive/smilealign/runs-smilealign-seg-v2`.
Path itu tidak ada di Kaggle, jadi dibuat symlink ke folder kerja supaya
Ultralytics menulis ke lokasi yang sama persis seperti di Colab.

In [ ]:
import os
from pathlib import Path

work = Path("/kaggle/working/smilealign")
work.mkdir(parents=True, exist_ok=True)

Path("/content/drive/MyDrive").mkdir(parents=True, exist_ok=True)
link = Path("/content/drive/MyDrive/smilealign")

if link.is_symlink():
    print("symlink sudah ada ->", os.path.realpath(link))
elif link.exists():
    print("PERINGATAN: ada folder biasa, bukan symlink:", link)
else:
    os.symlink(work, link)
    print("symlink dibuat ->", os.path.realpath(link))

RUNS.mkdir(parents=True, exist_ok=True)
print("isi RUNS:", [p.name for p in RUNS.iterdir()] or "(kosong)")

## Sel 3, unduh last.pt dan zip intraoral

In [ ]:
import shutil
from pathlib import Path

import gdown

assert LAST_PT_ID and ZIP_ID, "ID belum diisi di Sel 0"

STAGING.mkdir(parents=True, exist_ok=True)

# Kalau zip sudah diunggah sebagai Kaggle Dataset, tidak perlu internet.
pre_uploaded = sorted(Path("/kaggle/input").rglob("intraoral_yolov8.zip"))
zip_path = ROOT / "intraoral_yolov8.zip"

if not zip_path.exists():
    if pre_uploaded:
        shutil.copy2(pre_uploaded[0], zip_path)
        print("zip dari /kaggle/input:", pre_uploaded[0])
    else:
        gdown.download(id=ZIP_ID, output=str(zip_path), quiet=False)

size = zip_path.stat().st_size
print("zip     :", round(size / 1e6, 1), "MB")
assert size > 1e6, "zip terlalu kecil, unduhan gagal"

staged_last = STAGING / "last.pt"
if not staged_last.exists():
    gdown.download(id=LAST_PT_ID, output=str(staged_last), quiet=False)

size = staged_last.stat().st_size
print("last.pt :", round(size / 1e6, 1), "MB")
assert size > 1e6, "last.pt terlalu kecil, unduhan gagal"

## Sel 4, periksa checkpoint

Aman dijalankan berulang. Kalau `last.pt` sudah dipindah ke folder run pada
percobaan sebelumnya, sel ini memakai file di folder run, bukan unduhan lama di
staging.

In [ ]:
import shutil

import torch

staged = STAGING / "last.pt"
found = sorted(
    RUNS.glob("*/weights/last.pt"), key=lambda p: p.stat().st_mtime, reverse=True
)

if found:
    # Sudah ada hasil percobaan sebelumnya, itu yang paling baru.
    source = found[0]
    print("sumber :", source, "(dari percobaan sebelumnya)")
elif staged.exists():
    source = staged
    print("sumber :", source, "(unduhan baru)")
else:
    raise FileNotFoundError("last.pt tidak ada di staging maupun di RUNS")

ckpt = torch.load(source, map_location="cpu", weights_only=False)
args = ckpt["train_args"]

RUN_DIR = RUNS / (args.get("name") or "train")
weights = RUN_DIR / "weights"
weights.mkdir(parents=True, exist_ok=True)

target = weights / "last.pt"
if source != target and not target.exists():
    shutil.move(str(source), target)
    print("dipindah ke:", target)

print("folder run    :", RUN_DIR)
print("epoch terakhir:", ckpt["epoch"])
print("total epochs  :", args["epochs"])
print("sisa epoch    :", max(args["epochs"] - ckpt["epoch"], 0))
print("data          :", args["data"])
print("imgsz         :", args["imgsz"], "| batch:", args["batch"])
print("project       :", args["project"])
print("versi dibuat  :", ckpt.get("version"))
print("terpasang     :", __import__("ultralytics").__version__)

## Sel 5, selaraskan argumen checkpoint

Kalau versi terpasang berbeda dari pembuat checkpoint, `train(resume=True)`
menolak dengan pesan menyesatkan seperti `FileNotFoundError` atau
`SyntaxError: 'dis' is not a valid YOLO argument`.

Sel ini membandingkan `train_args` dengan `DEFAULT_CFG_DICT`:

- kunci yang tidak dikenal versi terpasang dibuang,
- nilai yang seharusnya bool tapi tersimpan sebagai float diubah jadi bool
  (kasus `multi_scale`),
- `last_orig.pt` disimpan sebagai cadangan dan tidak pernah ditimpa.

In [ ]:
import shutil

import torch
from ultralytics.cfg import DEFAULT_CFG_DICT

current = RUN_DIR / "weights" / "last.pt"
backup = RUN_DIR / "weights" / "last_orig.pt"

if not backup.exists():
    shutil.copy2(current, backup)
    print("cadangan dibuat :", backup.name)
else:
    shutil.copy2(backup, current)
    print("dipulihkan dari :", backup.name)

ckpt = torch.load(current, map_location="cpu", weights_only=False)
args = ckpt["train_args"]
known = set(DEFAULT_CFG_DICT)

unknown = sorted(k for k in args if k not in known)
for k in unknown:
    args.pop(k)

coerced = []
for k, v in list(args.items()):
    default = DEFAULT_CFG_DICT.get(k)
    if isinstance(default, bool) and not isinstance(v, bool):
        args[k] = bool(v)
        coerced.append(k + ": " + repr(v) + " -> " + repr(args[k]))

ckpt["train_args"] = args
torch.save(ckpt, current)

print("dibuang  :", unknown or "(tidak ada)")
print("disamakan:", coerced or "(tidak ada)")
print("ditulis  :", current)
print()
print("epoch terakhir:", ckpt["epoch"], "dari", args["epochs"])

## Sel 6, unduh dan ekstrak AlphaDent

1.455 foto gigi DSLR dari HuggingFace, Apache-2.0. Diunduh ulang karena
`/content` hilang setiap sesi berakhir.

In [ ]:
import os
import zipfile

from huggingface_hub import hf_hub_download

ALPHA_DIR = "/content/datasets/alphadent"
os.makedirs(ALPHA_DIR, exist_ok=True)

alpha_zip = hf_hub_download(
    repo_id="ZFTurbo/AlphaDent",
    filename="AlphaDent.zip",
    repo_type="dataset",
    local_dir="/content/ad_zip",
)
print("zip:", alpha_zip)

with zipfile.ZipFile(alpha_zip) as z:
    z.extractall(ALPHA_DIR)


def has_struct(root):
    a = os.path.isdir(os.path.join(root, "images", "train")) and os.path.isdir(
        os.path.join(root, "labels", "train")
    )
    b = os.path.isdir(os.path.join(root, "train", "images")) and os.path.isdir(
        os.path.join(root, "train", "labels")
    )
    return bool(a or b)


AD_ROOT = None
for root, dirs, files in os.walk(ALPHA_DIR):
    if has_struct(root):
        AD_ROOT = root
        break
assert AD_ROOT, "struktur AlphaDent tidak ketemu, cek isi zip"
print("AD_ROOT:", AD_ROOT)

## Sel 7, bangun dataset campuran

Logikanya sama dengan `scripts/prepare_v2_mix.py`. Semua kelas dipetakan ke
`tooth`, dan baris bbox AlphaDent diubah menjadi polygon karena Ultralytics
menolak file yang mencampur keduanya.

In [ ]:
import os
import shutil
import zipfile
from pathlib import Path

INTRAORAL_DIR = "/content/datasets/intraoral"
os.makedirs(INTRAORAL_DIR, exist_ok=True)

with zipfile.ZipFile(ROOT / "intraoral_yolov8.zip") as z:
    for info in z.infolist():
        name = info.filename.replace("\\", "/")
        target = os.path.join(INTRAORAL_DIR, name)
        if name.endswith("/"):
            os.makedirs(target, exist_ok=True)
            continue
        os.makedirs(os.path.dirname(target), exist_ok=True)
        with z.open(info) as src, open(target, "wb") as dst:
            dst.write(src.read())
print("intraoral:", os.listdir(os.path.join(INTRAORAL_DIR, "intraoral")))

JOBS = [
    ("io_", Path(INTRAORAL_DIR) / "intraoral"),
    ("ad_", Path(AD_ROOT)),
]
SPLITS = [("train", "train"), ("valid", "valid"), ("test", "valid")]


def pick_dirs(root, split):
    imgs = [p for p in (root / split / "images", root / "images" / split) if p.is_dir()]
    lbls = [p for p in (root / split / "labels", root / "labels" / split) if p.is_dir()]
    return (imgs[0] if imgs else None), (lbls[0] if lbls else None)


def normalize_row(line):
    # Bbox (4 angka) jadi polygon 4 titik. Polygon dibiarkan, kelas jadi 0.
    parts = line.split()
    if len(parts) < 2:
        return None
    coords = parts[1:]
    if len(coords) == 4:
        cx, cy, bw, bh = (float(v) for v in coords)
        x1, y1, x2, y2 = cx - bw / 2, cy - bh / 2, cx + bw / 2, cy + bh / 2
        corners = [(x1, y1), (x2, y1), (x2, y2), (x1, y2)]
        return "0 " + " ".join(format(v, ".6f") for pt in corners for v in pt)
    if len(coords) >= 6 and len(coords) % 2 == 0:
        return "0 " + " ".join(coords)
    return None


counts = {"train": 0, "valid": 0}
dropped = 0
for prefix, root in JOBS:
    for split_src, split_dst in SPLITS:
        imgs, lbls = pick_dirs(root, split_src)
        if imgs is None or lbls is None:
            continue
        (MIX_DIR / split_dst / "images").mkdir(parents=True, exist_ok=True)
        (MIX_DIR / split_dst / "labels").mkdir(parents=True, exist_ok=True)
        for img in sorted(imgs.iterdir()):
            if img.suffix.lower() not in (".jpg", ".jpeg", ".png"):
                continue
            lbl = lbls / (img.stem + ".txt")
            if not lbl.exists():
                continue
            rows = []
            for line in lbl.read_text().splitlines():
                t = line.strip()
                if not t:
                    continue
                row = normalize_row(t)
                if row is None:
                    dropped += 1
                    continue
                rows.append(row)
            if not rows:
                continue
            shutil.copy(img, MIX_DIR / split_dst / "images" / (prefix + img.name))
            (MIX_DIR / split_dst / "labels" / (prefix + img.stem + ".txt")).write_text(
                "\n".join(rows) + "\n"
            )
            counts[split_dst] += 1

(MIX_DIR / "data.yaml").write_text(
    "train: train/images\nval: valid/images\nnc: 1\nnames: ['tooth']\n"
)
print("merged:", counts, "->", MIX_DIR)
if dropped:
    print("baris label dibuang:", dropped)
assert (MIX_DIR / "data.yaml").exists()

## Sel 8, lanjutkan latihan

Tanpa argumen tambahan. `resume=True` membaca ulang semua pengaturan dari
checkpoint, termasuk folder output, jadi latihan lanjut di folder yang sama.

Bisa berhenti sendiri sebelum epoch terakhir karena `patience=20`. Itu normal,
`best.pt` tetap tersimpan dari epoch terbaik.

In [ ]:
import os

os.environ["WANDB_MODE"] = "disabled"
os.environ["COMET_MODE"] = "disabled"

from ultralytics import YOLO

data_yaml = Path(ckpt["train_args"]["data"])
assert data_yaml.exists(), "dataset tidak ada: " + str(data_yaml) + ", ulangi Sel 7"
assert (RUN_DIR / "weights" / "last.pt").exists(), "last.pt hilang, ulangi Sel 4"

model = YOLO(str(RUN_DIR / "weights" / "last.pt"))
model.train(resume=True)

## Sel 9, ekspor hasil

In [ ]:
import shutil
from pathlib import Path

out = Path("/kaggle/working/export")
out.mkdir(parents=True, exist_ok=True)

for name in ("best.pt", "last.pt"):
    src = RUN_DIR / "weights" / name
    if src.exists():
        shutil.copy(src, out / name)
        print(name, "->", round(src.stat().st_size / 1e6, 1), "MB")

csv = RUN_DIR / "results.csv"
if csv.exists():
    shutil.copy(csv, out / "results.csv")

print()
print("folder:", out)
print("Unduh best.pt, taruh di backend/models/seg/best.pt")

## Kalau gagal

| Gejala | Sebab | Tindakan |
|---|---|---|
| `No module named 'ultralytics'` | Sel 1 belum jalan | jalankan Sel 1 |
| `'dis' is not a valid YOLO argument` | versi beda dari checkpoint | jalankan Sel 5 |
| `'multi_scale=0.0' is of invalid type float` | idem, kasus bool | jalankan Sel 5 |
| `FileNotFoundError` data.yaml | Sel 7 belum selesai | ulangi Sel 7 |
| `cuda: False` | Accelerator belum GPU | Settings, restart sesi |
| `nvidia-smi: command not found` | Accelerator masih None | Settings, restart sesi |

## Kalau sesi mati

Ulangi Sel 0 sampai Sel 4. Sel 6 dan Sel 7 perlu diulang karena `/content`
hilang. Kalau Drive belum terbarui, ambil `last.pt` terbaru dari panel Output
notebook ini, unggah ke Drive, lalu pakai ID yang baru.